# I3. The MeerKAT array

The third notebook of the introductory track ({doc}`../index`). It reads
MeerKAT's 64 antenna positions, shipped with rang as package data
(`rang/data/meerkat.itrf.txt`, from simms; {gh}`data/README.md`), and the
public fixture of I2. No simulation is written; it runs in under a minute.

**Learning goals.** After this notebook you can

- describe MeerKAT's layout: 64 dishes of 13.5 m, a dense core and arms
  that give baselines up to 7.7 km, and count its baselines;
- explain Earth-rotation synthesis and how the uv tracks depend on
  declination;
- compute the angular resolution λ/B_max and the field of view, set by the
  primary beam of one dish;
- say what four antennas (the fixture), sixteen (A1 small) and 64 give in uv
  coverage and point spread function.

In [ ]:
%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units
from astropy.coordinates import EarthLocation
from IPython.display import display

from rang import simtruth as st
from rang.notebook import data
from rang.notebook import image as nbimage

C = 299_792_458.0
DEG = math.pi / 180
NU0 = 1.284e9  # the centre of L band, 856-1712 MHz

xyz, diameter, names = st.antenna_table()
print(f"{len(names)} antennas ({names[0]} to {names[-1]}), dish diameters {sorted(set(diameter))} m")
print("source of the table:", st.ITRF.relative_to(st.ITRF.parents[2]))

## 1. The layout

The table holds Earth-centred (ITRF) coordinates in metres. Rotated to east
and north at the array centre:

In [ ]:
centre = xyz.mean(axis=0)
site = EarthLocation.from_geocentric(*centre, unit=units.m)
lat, lon = site.lat.rad, site.lon.rad
east_axis = np.array([-math.sin(lon), math.cos(lon), 0.0])
north_axis = np.array([-math.sin(lat) * math.cos(lon), -math.sin(lat) * math.sin(lon), math.cos(lat)])
enu = np.stack([(xyz - centre) @ east_axis, (xyz - centre) @ north_axis], -1)
radius = np.hypot(*enu.T)
print(f"array centre: latitude {site.lat.deg:.3f} deg, longitude {site.lon.deg:.3f} deg, height {site.height.to_value(units.m):.0f} m")
print(f"antennas within 0.5 km of the centre: {(radius < 500).sum()}; within 1 km: {(radius < 1000).sum()}; farthest {radius.max() / 1e3:.2f} km")

small = st.design("small").antennas  # A1 small: every fourth antenna
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, lim in zip(axes, (4.5, 1.2)):
    ax.scatter(*enu.T / 1e3, s=12, label="64 antennas")
    ax.scatter(*enu[small].T / 1e3, s=40, facecolors="none", edgecolors="C3", label="A1 small (16)")
    ax.scatter(*enu[:4].T / 1e3, marker="x", color="k", label="fixture (m000-m003)")
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel="east (km)", ylabel="north (km)", aspect="equal")
axes[0].set_title("MeerKAT")
axes[1].set_title("the core")
axes[1].legend(loc="lower left", fontsize=8)
plt.show()

The positions in the table and those in a real observation's ANTENNA table
can differ. The fixture's own ANTENNA table against this one, for its four
antennas:

In [ ]:
fixture = data.load_fixture(data.find_fixture() or data.download_fixture(data.default_cache_dir()))
fx = fixture.extra["antenna_position"]
offset = np.linalg.norm(fx - xyz[:4], axis=1)
relative = np.linalg.norm((fx - fx[0]) - (xyz[:4] - xyz[0]), axis=1)
print(f"absolute offsets {np.round(offset, 2)} m; relative to m000 {np.round(relative * 100, 1)} cm")

The two tables differ by a nearly common shift, which leaves baselines (the
differences) the same to centimetres: an interferometer measures baselines,
not absolute positions. A1 uses the table as it is.

## 2. Baselines

n antennas give n(n − 1)/2 baselines:

In [ ]:
pairs = np.array([(p, q) for p in range(len(xyz)) for q in range(p + 1, len(xyz))])
lengths = np.linalg.norm(xyz[pairs[:, 0]] - xyz[pairs[:, 1]], axis=1)
pd.DataFrame(
    [
        {
            "array": label,
            "antennas": len(ants),
            "baselines": len(ants) * (len(ants) - 1) // 2,
            "shortest (m)": round(float(np.min(ls)), 1),
            "longest (m)": round(float(np.max(ls)), 1),
        }
        for label, ants in (("fixture", range(4)), ("A1 small", small), ("MeerKAT", range(64)))
        for ls in [[np.linalg.norm(xyz[p] - xyz[q]) for i, p in enumerate(ants) for q in list(ants)[i + 1 :]]]
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(lengths / 1e3, bins=np.logspace(1, np.log10(8e3), 40) / 1e3)
ax.set(xscale="log", xlabel="baseline length (km)", ylabel="baselines", title="All 2,016 MeerKAT baselines")
plt.show()
print(f"baselines shorter than 1 km: {(lengths < 1000).mean():.0%}")

## 3. Resolution and field of view

The finest structure an array resolves is about λ / B_max; the field of view
is the primary beam of one dish, about λ / D wide. rang's beam module reads
MeerKAT's L-band beam model (katbeam's MKAT-AA-L-JIM-2020; the `beam` and
`pointing` extras):

In [ ]:
b_max = lengths.max()
for f in (856e6, NU0, 1712e6):
    lam = C / f
    print(f"{f / 1e6:.0f} MHz: lambda = {lam * 100:.1f} cm, lambda/B_max = {lam / b_max / DEG * 3600:.1f} arcsec, lambda/D = {lam / 13.5 / DEG * 60:.0f} arcmin")

try:
    from rang import beams

    table, meta = beams.load_katbeam("H")
except ImportError as error:
    table = None
    print("skipped (install the beam and pointing extras):", error)

In [ ]:
if table is not None:
    f_tab = np.asarray(table.frequency_hz)
    fwhm = np.degrees(np.asarray(table.fwhm_rad)).mean(axis=1) * 60  # arcmin, mean of the two axes
    k = fwhm / 60 * DEG / (C / f_tab / 13.5)
    print(f"{meta['model']}: power FWHM {np.interp(NU0, f_tab, fwhm):.1f} arcmin at 1284 MHz = {np.interp(NU0, f_tab, k):.2f} lambda/D")
    r = np.linspace(0, 1.6, 300) * DEG
    fig, ax = plt.subplots(figsize=(6, 3))
    for f in (900e6, NU0, 1650e6):
        v = np.asarray(beams.voltage_beam(r[None, :], np.zeros((1, len(r))), np.array([f]), table))[0]
        ax.plot(np.degrees(r), 10 * np.log10(np.maximum(v**2, 1e-6)), label=f"{f / 1e6:.0f} MHz")
    ax.set(xlabel="distance from the pointing centre (deg)", ylabel="power beam (dB)", ylim=(-40, 1), title="Cosine-taper model with katbeam widths")
    ax.legend()
    plt.show()

Arcseconds of resolution over a field of about a degree: an image of the
full primary beam has thousands of resolution elements across.

## 4. Earth-rotation synthesis

Each baseline is one point in the (u, v) plane at one instant. As the Earth
rotates, the baseline's projection onto the plane perpendicular to the
source turns and shortens, and the point traces an ellipse. For a baseline
(L_x, L_y, L_z) in Earth-centred axes, Greenwich hour angle H and
declination δ:

u = sin H L_x + cos H L_y,  v = −sin δ cos H L_x + sin δ sin H L_y + cos δ L_z.

Eight hours of A1 small's 16 antennas at three declinations, in
kilo-wavelengths at 1284 MHz:

In [ ]:
def uv_tracks(ants, dec_deg, hours=8.0, steps=97, freq=NU0):
    """(time, baseline, 2) u, v in wavelengths over `hours` around transit."""
    ants = list(ants)
    bl = np.array([xyz[p] - xyz[q] for i, p in enumerate(ants) for q in ants[i + 1 :]])
    h = np.linspace(-hours / 2, hours / 2, steps) * 15 * DEG - lon  # local hour angle 0 at the centre
    dec = dec_deg * DEG
    sh, ch = np.sin(h)[:, None], np.cos(h)[:, None]
    u = sh * bl[:, 0] + ch * bl[:, 1]
    v = -math.sin(dec) * ch * bl[:, 0] + math.sin(dec) * sh * bl[:, 1] + math.cos(dec) * bl[:, 2]
    return np.stack([u, v], -1) * freq / C


fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, dec in zip(axes, (-30, -60, 0)):
    uv = uv_tracks(small, dec) / 1e3
    for sign in (1, -1):  # V(-u, -v) is the complex conjugate of V(u, v): each baseline gives two tracks
        ax.plot(sign * uv[..., 0], sign * uv[..., 1], "-", color="C0", lw=0.4)
    ax.set(title=f"declination {dec} deg", xlabel="u (kλ)", ylabel="v (kλ)", aspect="equal")
plt.show()

Each track is part of an ellipse with axis ratio |sin δ|: nearly circular
towards the pole (δ = −60°: 0.87), flattened to lines on the celestial
equator, which gives a beam elongated north-south. Eight hours cover a third
of each ellipse; a full track takes 12 h of a source that stays up (at
MeerKAT's latitude, −30.7°, every source south of δ = −59.3°).

## 5. What 4, 16 and 64 antennas give

The **point spread function** (PSF, the dirty beam) is the image of a point
source made from the sampled (u, v): the Fourier transform of the sampling.
One hour around transit at δ = −30°, 1284 MHz, natural weighting, by direct
Fourier sum on a 2-arcminute field:

In [ ]:
grid = np.linspace(-120, 120, 121) / 3600 * DEG
ll, mm = np.meshgrid(grid, grid)
psfs = {}
for label, ants in (("fixture, 4", range(4)), ("A1 small, 16", small), ("MeerKAT, 64", range(64))):
    uv = uv_tracks(ants, -30, hours=1.0, steps=7 if label.startswith("Meer") else 31).reshape(-1, 2)
    psf = np.zeros_like(ll)
    for chunk in np.array_split(uv, max(1, len(uv) // 500)):
        psf += np.cos(2 * np.pi * (chunk[:, 0, None, None] * ll + chunk[:, 1, None, None] * mm)).sum(axis=0)
    psfs[label] = psf / len(uv)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, (label, psf) in zip(axes, psfs.items()):
    im = ax.imshow(psf, origin="lower", extent=[120, -120, -120, 120], cmap="RdBu_r", vmin=-0.3, vmax=1)
    ax.set(title=label, xlabel="RA offset (arcsec)", ylabel="Dec offset (arcsec)")
fig.colorbar(im, ax=axes, shrink=0.8, label="PSF")
plt.show()
pd.DataFrame(
    [
        {"array": label, "peak sidelobe beyond 30 arcsec": f"{np.abs(psf[np.hypot(ll, mm) > 30 / 3600 * DEG]).max():.3f}"}
        for label, psf in psfs.items()
    ]
)

Four core antennas give a PSF tens of arcminutes wide (I2): within this
field it is one smooth blob. Sixteen with baselines up to 7.7 km resolve arcseconds,
but with strong sidelobes. 64 fill the uv plane and lower the sidelobes
further. Deconvolution (I7) removes what sidelobes remain. The fixture's own
coverage, from its UVW column, for comparison:

In [ ]:
display(nbimage.uv_coverage(fixture))

## Exercises

1. At which declination does a MeerKAT source never rise? Use the latitude
   above.
2. Plot the uv tracks for all 64 antennas at δ = −30° over 8 h. Where is the
   coverage sparse, and which baselines fill it?
3. The PSF of section 5 at δ = 0°: how much wider is it north-south than
   east-west?
4. A source of 2 arcmin at 1.4 GHz: which MeerKAT baselines still see more
   than half its flux (I2, the Gaussian visibility)?

## Where next

- {doc}`i4-measurement-equation`: what the instrument does to each visibility.
- Notebook {doc}`04-imaging` makes the dirty image and PSF of the fixture
  and compares weightings; {doc}`../../concepts/imaging`.
- {doc}`i2-visibilities` for the two-element interferometer.